# ReID — Held-out evaluation (1× GPU, `.reidraw` targets)

Evaluates trained runs from the team Hugging Face bucket on the Kaggle unseen-target bundle
[`a7medlabib/reid-unseen-evaluation-targets-v1`](https://www.kaggle.com/datasets/a7medlabib/reid-unseen-evaluation-targets-v1).

**Flow**
1. Scan bucket folders (`v3_50epoch_28c68a1`, `v3_50epoch_b69c47a`, `v3_50epoch_56923ec`) and read every `latest.json`.
2. A run is **[PASS]** only when `global_steps == 100000` (all 50 epochs × 2,000 steps finished).
3. Download the chosen run's checkpoint, verify its SHA-256 against `best.json` / `latest.json`, print exactly which file was downloaded.
4. Evaluate on the 5 public benchmarks (Market1501, MSMT17, MARS, DukeMTMC, Occluded-Duke). `TeamRecorded` is skipped.

**Config `smoke_test`** evaluates a single model and reports how long each stage takes, so we can budget the full 27-run sweep.

**Outputs per dataset:** Rank-1, Rank-5, mAP, CMC ranks 1–20 (%). **Macro:** unweighted mean over the five benchmarks.

**Attach on Kaggle:** input `a7medlabib/reid-unseen-evaluation-targets-v1`; secrets `GITHUB_TOKEN`, `HF_TOKEN`; 1× T4 / P100.

Code is checked out from Git branch **`main`** (same repo as training notebooks).

In [ ]:
# ---- edit these ----
EVAL_CONFIG = "smoke_test"     # "smoke_test" = one model + timing report
CONDITION = None               # e.g. "B6"; None = first run that finished 50 epochs
SEED = None                    # e.g. 0;    None = first finished seed of CONDITION
CHECKPOINT = "best"            # "best" -> model_best.pth (paper protocol) | "final" -> 50-epoch resume slot
DATASETS = ["market1501", "msmt17", "mars", "dukemtmc", "occludedduke"]
RESEARCH_BRANCH = "main"
HF_BUCKET_ID = "reid-research-hub/reid-loss-study-live"
BUCKET_FOLDERS = ["v3_50epoch_28c68a1", "v3_50epoch_b69c47a", "v3_50epoch_56923ec"]
TARGET_GLOBAL_STEPS = 50 * 2000
TOTAL_RUNS = 27                # 9 conditions x 3 seeds, for the time estimate
REPO_URL = "https://github.com/BASSAT-BASSAT/Person-Re-ID-Foundational-model.git"
# --------------------

import hashlib
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import torch
from kaggle_secrets import UserSecretsClient

assert CHECKPOINT in ("best", "final"), CHECKPOINT
ROOT = Path("/kaggle/working/Person-Re-ID-Foundational-model-eval")
WORK_ROOT = Path("/kaggle/working/paper_eval")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_CANDIDATES = [
    Path("/kaggle/input/reid-unseen-evaluation-targets-v1"),
    Path("/kaggle/input/datasets/a7medlabib/reid-unseen-evaluation-targets-v1"),
    Path("/kaggle/input/a7medlabib/reid-unseen-evaluation-targets-v1"),
]
TARGET_ROOT = next((p for p in TARGET_CANDIDATES if (p / "reid_targets_index.json").is_file()), None)
assert TARGET_ROOT is not None, "Attach dataset a7medlabib/reid-unseen-evaluation-targets-v1"

assert torch.cuda.is_available(), "Enable one GPU"
secret_client = UserSecretsClient()
GITHUB_TOKEN = secret_client.get_secret("GITHUB_TOKEN")
HF_TOKEN = secret_client.get_secret("HF_TOKEN")
assert GITHUB_TOKEN and HF_TOKEN, "Enable GITHUB_TOKEN and HF_TOKEN in Kaggle Secrets"
TIMER = {}
print(f"[PASS] GPU {torch.cuda.get_device_name(0)} | targets {TARGET_ROOT} | config {EVAL_CONFIG}")

In [ ]:
t0 = time.perf_counter()
if not ROOT.is_dir():
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
    subprocess.check_call(["git", "clone", "--quiet", clone_url, str(ROOT)])
subprocess.check_call(["git", "fetch", "--quiet", "origin", RESEARCH_BRANCH], cwd=ROOT)
subprocess.check_call(
    ["git", "checkout", "--quiet", "-B", RESEARCH_BRANCH, f"origin/{RESEARCH_BRANCH}"],
    cwd=ROOT,
)
sys.path.insert(0, str(ROOT))
rev = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
subprocess.check_call([sys.executable, str(ROOT / "scripts/setup_fastreid.py"), "--cython"], cwd=ROOT)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]", "huggingface_hub>=1.7.1,<2"], cwd=ROOT)
assert (ROOT / "scripts/evaluate_unseen_reidraw.py").is_file(), f"Branch {RESEARCH_BRANCH} missing eval scripts"
TIMER["setup_s"] = time.perf_counter() - t0
print(f"[PASS] Repo origin/{RESEARCH_BRANCH} @ {rev[:12]} ({TIMER['setup_s']:.0f}s)")

### Hotfix (only if `git pull` is still on an old commit)

Run **once** before §4 Evaluate. Overwrites `scripts/evaluate_unseen_reidraw.py` so eval never loads `pool_v2.yaml`. Skip after `main` @ `afcf896+`.


In [ ]:
%%writefile /kaggle/working/Person-Re-ID-Foundational-model-eval/scripts/evaluate_unseen_reidraw.py
"""Evaluate a trained checkpoint on Kaggle unseen ``.reidraw`` targets (V3 protocol)."""

from __future__ import annotations

import argparse
import logging
import sys
import time
from collections import OrderedDict
from pathlib import Path

REPO_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(REPO_ROOT))

from reidfm.data.unseen_targets import (  # noqa: E402
    find_target_root,
    iter_target_records,
    load_target_specs,
    reidraw_files_by_root_id,
)
from reidfm.experiments.contract import HELD_OUT_TEST_KEYS  # noqa: E402
from reidfm.experiments.loss_study import expand_loss_study_run  # noqa: E402
from reidfm.fastreid_compat import ensure_fastreid  # noqa: E402


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--checkpoint", required=True, help="model_best.pth or model_final.pth")
    parser.add_argument("--condition", default="", help="loss-study condition (B6, R0, …) to rebuild cfg")
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument(
        "--config-file",
        default=None,
        help="training config.yaml (default: beside checkpoint; else loss-study base if --condition)",
    )
    parser.add_argument("--target-root", default="", help="folder with *.reidraw + reid_targets_index.json")
    parser.add_argument("--name", default="")
    parser.add_argument("--out", default="results/paper_eval")
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--workers", type=int, default=4)
    parser.add_argument("--device", default="cuda")
    parser.add_argument("--max-rank", type=int, default=50)
    parser.add_argument(
        "--num-classes", type=int, default=10036,
        help="training identities; SphereFace2 / angular heads need the real class count",
    )
    parser.add_argument(
        "--datasets", default=",".join(HELD_OUT_TEST_KEYS),
        help="comma-separated subset of held-out keys",
    )
    return parser.parse_args()


def _result_key_to_battery(key: str) -> str | None:
    dataset = key.split("|", 1)[0].strip().lower().replace("-", "").replace("_", "")
    aliases = {
        "market1501": "market1501",
        "msmt17": "msmt17",
        "mars": "mars",
        "dukemtmc": "dukemtmc",
        "occludeddukemtmc": "occludedduke",
        "occludedduke": "occludedduke",
    }
    return aliases.get(dataset)


def _resolve_eval_config(args, checkpoint: Path):
    """Pick config path + opts without requiring pool_v2.yaml on disk."""
    from reidfm.utils.paths import repo_path

    explicit = Path(args.config_file) if args.config_file else None
    ckpt_cfg = checkpoint.parent / "config.yaml"
    opts: list[str] = ["MODEL.DEVICE", args.device, "MODEL.HEADS.NUM_CLASSES", str(args.num_classes)]
    model_key = None
    config_file: Path | None = None

    if args.condition:
        run = expand_loss_study_run(args.condition, args.seed)
        opts = run.opts + opts
        model_key = run.init_model
        if explicit and explicit.is_file():
            config_file = explicit
        elif ckpt_cfg.is_file():
            config_file = ckpt_cfg
        else:
            config_file = repo_path(run.config_file)
    elif explicit and explicit.is_file():
        config_file = explicit
    elif ckpt_cfg.is_file():
        config_file = ckpt_cfg
    else:
        raise FileNotFoundError(
            f"Missing config beside checkpoint ({ckpt_cfg}); pass --condition/--seed or --config-file."
        )
    return config_file, opts, model_key


def setup_eval_cfg_for_heldout(
    config_file,
    opts=None,
    model_key=None,
    output_dir=None,
    num_classes=10036,
):
    """Eval-only cfg: never loads ``pool_v2.yaml`` (works on old clones without ``setup_eval_cfg`` in config)."""
    import reidfm.config as config_mod

    if hasattr(config_mod, "setup_eval_cfg"):
        return config_mod.setup_eval_cfg(
            config_file,
            opts=opts,
            model_key=model_key,
            output_dir=output_dir,
            num_classes=num_classes,
        )
    from reidfm.config import apply_init_model, load_cfg, resolve_output_dir

    cfg = load_cfg(config_file, opts)
    cfg.DATASETS.NAMES = ()
    cfg.DATASETS.TESTS = ()
    if int(cfg.MODEL.HEADS.NUM_CLASSES) <= 0:
        cfg.MODEL.HEADS.NUM_CLASSES = int(num_classes)
    apply_init_model(cfg, model_key, prepare_weights=False)
    resolve_output_dir(cfg, output_dir)
    cfg.freeze()
    return cfg


def main() -> int:
    args = parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
    ensure_fastreid()
    from reidfm.evaluation.paper_report import build_paper_payload, format_paper_markdown
    from reidfm.evaluation.v3_evaluator import evaluate_v3_packed_groups
    from reidfm.evaluation.extractors import build_fastreid_model
    from reidfm.utils.paths import repo_path
    from reidfm.utils.provenance import run_provenance, write_json

    checkpoint = Path(args.checkpoint)
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)

    target_root = Path(args.target_root) if args.target_root else find_target_root()
    specs = load_target_specs(target_root)
    reidraw_map = reidraw_files_by_root_id(specs)

    datasets = [key.strip() for key in args.datasets.split(",") if key.strip()]
    unknown = [key for key in datasets if key not in HELD_OUT_TEST_KEYS]
    if unknown:
        raise ValueError(f"Unknown held-out keys {unknown}; choose from {list(HELD_OUT_TEST_KEYS)}")

    config_file, opts, model_key = _resolve_eval_config(args, checkpoint)
    cfg = setup_eval_cfg_for_heldout(
        config_file,
        opts=opts,
        model_key=model_key,
        output_dir=str(checkpoint.parent),
        num_classes=args.num_classes,
    )

    started = time.perf_counter()
    model, cfg = build_fastreid_model(cfg, checkpoint, args.device)
    size_hw = tuple(int(x) for x in cfg.INPUT.SIZE_TEST)
    timings = OrderedDict(model_load_s=round(time.perf_counter() - started, 2))

    raw = OrderedDict()
    for key in datasets:
        records = list(iter_target_records(specs[key], target_root))
        n_query = sum(row.get("retrieval_role") == "query" for row in records)
        print(f"[{key}] {n_query} query / {len(records) - n_query} gallery images", flush=True)
        t0 = time.perf_counter()
        part = evaluate_v3_packed_groups(
            model,
            records,
            target_root,
            size_hw=size_hw,
            batch_size=args.batch_size,
            num_workers=args.workers,
            device=args.device,
            max_rank=args.max_rank,
            reidraw_files=reidraw_map,
        )
        timings[key] = round(time.perf_counter() - t0, 2)
        print(f"[{key}] done in {timings[key]:.1f}s", flush=True)
        raw.update((k, v) for k, v in part.items() if k != "mean")
    timings["total_s"] = round(time.perf_counter() - started, 2)

    battery_results = OrderedDict()
    for proto_key, metrics in raw.items():
        if proto_key == "mean":
            continue
        battery_key = _result_key_to_battery(proto_key)
        if battery_key is None:
            continue
        battery_results[battery_key] = {
            "Rank-1": metrics["Rank-1"],
            "Rank-5": metrics["Rank-5"],
            "Rank-10": metrics.get("Rank-10", metrics["Rank-5"]),
            "Rank-20": metrics.get("Rank-20", metrics["Rank-5"]),
            "mAP": metrics["mAP"],
            "mINP": metrics.get("mINP", 0.0),
            "cmc": list(metrics.get("cmc") or []),
        }
        if len(battery_results[battery_key]["cmc"]) < 20:
            cmc = battery_results[battery_key]["cmc"]
            last = cmc[-1] if cmc else battery_results[battery_key]["Rank-20"]
            battery_results[battery_key]["cmc"] = cmc + [last] * (20 - len(cmc))

    paper = build_paper_payload(battery_results)
    name = args.name or f"{args.condition or checkpoint.parent.name}_heldout"
    out_dir = repo_path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)
    markdown = format_paper_markdown(paper, title=f"{name} — unseen targets (reidraw)")
    print(markdown)
    write_json(
        out_dir / f"{name}.json",
        {
            "name": name,
            "checkpoint": str(checkpoint),
            "target_root": str(target_root),
            "condition": args.condition,
            "seed": args.seed,
            "datasets": datasets,
            "timings_s": timings,
            "protocol_results": raw,
            "battery_results": battery_results,
            "paper": paper,
            "provenance": run_provenance(),
        },
    )
    (out_dir / f"{name}.md").write_text(markdown, encoding="utf-8")
    print("timings (s): " + ", ".join(f"{k}={v}" for k, v in timings.items()))
    print(f"wrote {out_dir / name}.json and .md")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## 1. Which runs finished 50 epochs?

Reads `<folder>/<condition>/seed<N>/latest.json` for every run in both folders.
`v3_50epoch_56923ec` was trained before the loss fixes, so only R0, B2 and B3 (whose code did not change) are accepted from it.

In [ ]:
import tempfile
from huggingface_hub import download_bucket_files, list_bucket_tree

UNCHANGED_SINCE_56923EC = {"R0", "B2", "B3"}


def bucket_files(prefix):
    return {
        item.path for item in list_bucket_tree(HF_BUCKET_ID, prefix=prefix, recursive=True, token=HF_TOKEN)
        if getattr(item, "type", "file") == "file"
    }


def read_bucket_json(key):
    with tempfile.TemporaryDirectory() as td:
        dst = Path(td) / "doc.json"
        download_bucket_files(HF_BUCKET_ID, files=[(key, dst)], raise_on_missing_files=True, token=HF_TOKEN)
        return json.loads(dst.read_text(encoding="utf-8"))


runs = []
for folder in BUCKET_FOLDERS:
    keys = bucket_files(folder)
    for key in sorted(k for k in keys if k.endswith("/latest.json")):
        parts = key.split("/")
        if len(parts) != 4 or not parts[2].startswith("seed"):
            continue
        _, condition, seed_dir, _ = parts
        prefix = "/".join(parts[:3])
        doc = read_bucket_json(key)
        steps = int(doc.get("global_steps", -1))
        has_best = f"{prefix}/model_best.pth" in keys and f"{prefix}/best.json" in keys
        if steps < TARGET_GLOBAL_STEPS:
            status = "TRAINING"
        elif folder.endswith("56923ec") and condition not in UNCHANGED_SINCE_56923EC:
            status = "STALE"   # trained with pre-fix loss code
        elif CHECKPOINT == "best" and not has_best:
            status = "NO_BEST"
        else:
            status = "PASS"
        runs.append({
            "folder": folder, "condition": condition, "seed": int(seed_dir[4:]), "prefix": prefix,
            "global_steps": steps, "epoch": doc.get("epoch"), "status": status, "latest": doc,
            "research_commit": str(doc.get("research_commit", ""))[:12],
        })

print(f"{'folder':<20} {'cond':<5} {'seed':<4} {'steps':>7} {'epoch':>5}  {'commit':<12} status")
for r in runs:
    print(f"{r['folder']:<20} {r['condition']:<5} {r['seed']:<4} {r['global_steps']:>7} "
          f"{str(r['epoch']):>5}  {r['research_commit']:<12} [{r['status']}]")
finished = [r for r in runs if r["status"] == "PASS"]
print(f"\n{len(finished)}/{len(runs)} runs finished all 50 epochs ({TARGET_GLOBAL_STEPS} steps)")
assert finished, "No run has reached 50 epochs yet"

## 2. Download one finished model

In [ ]:
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        while block := stream.read(chunk):
            h.update(block)
    return h.hexdigest()


candidates = [
    r for r in finished
    if (CONDITION is None or r["condition"] == CONDITION) and (SEED is None or r["seed"] == SEED)
]
assert candidates, f"{CONDITION} seed {SEED} has not finished 50 epochs (see table above)"
RUN = candidates[0]
CONDITION, SEED = RUN["condition"], RUN["seed"]
WORK = WORK_ROOT / EVAL_CONFIG / f"{CONDITION}_seed{SEED}"
WORK.mkdir(parents=True, exist_ok=True)

if CHECKPOINT == "best":
    meta = read_bucket_json(f"{RUN['prefix']}/best.json")
    remote_ckpt = f"{RUN['prefix']}/model_best.pth"
    local_ckpt = WORK / "model_best.pth"
else:
    meta = RUN["latest"]
    remote_ckpt = meta["remote_file"]
    local_ckpt = WORK / meta["filename"]

t0 = time.perf_counter()
if not (local_ckpt.is_file() and local_ckpt.stat().st_size == meta.get("size")):
    download_bucket_files(HF_BUCKET_ID, files=[(remote_ckpt, local_ckpt)], raise_on_missing_files=True, token=HF_TOKEN)
TIMER["download_s"] = time.perf_counter() - t0
assert local_ckpt.stat().st_size == meta["size"], "Size mismatch against bucket metadata"
assert sha256(local_ckpt) == meta["sha256"], "SHA-256 mismatch against bucket metadata"

print("[PASS] latest.json: run finished all 50 epochs")
print(f"[PASS] Downloaded model : {HF_BUCKET_ID}/{remote_ckpt}")
print(f"       condition / seed : {CONDITION} / seed {SEED}")
print(f"       bucket folder    : {RUN['folder']}  (trained at commit {RUN['research_commit']})")
print(f"       global steps     : {RUN['global_steps']}  (epoch {RUN['epoch']})")
if CHECKPOINT == "best":
    print(f"       best epoch index : {meta.get('best_epoch_index')}  (val metric {meta.get('validation_metric')})")
print(f"       size / sha256    : {local_ckpt.stat().st_size / 1024**2:.1f} MB / {meta['sha256'][:16]}")
print(f"       download time    : {TIMER['download_s']:.1f}s")

## 3. Check the target bundle

Prints the bundle layout so a loader problem is easy to diagnose, then confirms every benchmark resolves to query/gallery records.

In [ ]:
from reidfm.data.unseen_targets import iter_target_records, load_target_specs

for path in sorted(TARGET_ROOT.iterdir()):
    print(f"  {path.name:<40} {path.stat().st_size / 1024**2:>10.1f} MB")
index_text = (TARGET_ROOT / "reid_targets_index.json").read_text(encoding="utf-8")
print("\nreid_targets_index.json (first 1500 chars):\n" + index_text[:1500])

specs = load_target_specs(TARGET_ROOT)
for key in DATASETS:
    records = list(iter_target_records(specs[key], TARGET_ROOT))
    n_query = sum(r.get("retrieval_role") == "query" for r in records)
    assert n_query and len(records) > n_query, f"{key}: empty query or gallery"
    print(f"[PASS] {key:<13} {specs[key]['reidraw_path'].name:<26} query {n_query:>6}  gallery {len(records) - n_query:>7}")

## 4. Evaluate (timed)

In [ ]:
name = f"{EVAL_CONFIG}_{CONDITION}_seed{SEED}_{CHECKPOINT}"
cmd = [
    sys.executable,
    str(ROOT / "scripts/evaluate_unseen_reidraw.py"),
    "--checkpoint", str(local_ckpt),
    "--condition", CONDITION,
    "--seed", str(SEED),
    "--target-root", str(TARGET_ROOT),
    "--datasets", ",".join(DATASETS),
    "--name", name,
    "--out", str(WORK / "results"),
    "--device", "cuda",
    "--batch-size", "128",
    "--workers", "4",
]
t0 = time.perf_counter()
subprocess.check_call(cmd, cwd=ROOT)
TIMER["evaluate_s"] = time.perf_counter() - t0
print(f"[PASS] Evaluated {CONDITION} seed {SEED} in {TIMER['evaluate_s'] / 60:.1f} min -> {WORK / 'results'}")

In [ ]:
report = json.loads((WORK / "results" / f"{name}.json").read_text(encoding="utf-8"))
print((WORK / "results" / f"{name}.md").read_text(encoding="utf-8"))
macro = report["paper"]["macro_public"]
print(f"Macro ({len(report['paper']['per_dataset'])} public): "
      f"R1={macro['Rank-1']:.2f}%  R5={macro['Rank-5']:.2f}%  mAP={macro['mAP']:.2f}%")

## 5. Smoke-test timing

In [ ]:
stages = report["timings_s"]
print(f"Model evaluated : {HF_BUCKET_ID}/{remote_ckpt}")
print(f"{'stage':<16} {'seconds':>9}")
for key in ("setup_s", "download_s"):
    if key in TIMER:
        print(f"{key:<16} {TIMER[key]:>9.1f}")
for key, value in stages.items():
    print(f"{key:<16} {value:>9.1f}")
per_model = TIMER.get("download_s", 0.0) + TIMER["evaluate_s"]
print(f"\nPer model (download + evaluate): {per_model / 60:.1f} min")
print(f"Estimate for {TOTAL_RUNS} runs on this GPU: {per_model * TOTAL_RUNS / 3600:.1f} h "
      f"(+ one-time setup {TIMER.get('setup_s', 0) / 60:.1f} min)")